In [132]:
import numpy as np
from typing import Dict, List, Tuple, Optional
from abc import ABC, abstractmethod
from enum  import Enum
from dataclasses import dataclass
import polars as pl
from scipy.stats import spearmanr, ks_2samp
from scipy.stats import entropy as scipy_entropy
from sklearn.feature_selection import mutual_info_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor


In [133]:
MICROSECOND = 1
MILLISECOND = 1_000 * MICROSECOND
SECOND = 1_000 * MILLISECOND

TWO_WEEKS_SEC = 14 * 24 * 60 * 60
BYBIT_LIQUIDATION_DELAY_US = 200 * MILLISECOND


**`ParquetDataLoader`** - Loads and slices multiple parquet feeds by microsecond `timestamp`. Supports custom ranges, first-week/two-week presets, and fixed-size chunked iteration.


In [134]:
class ParquetDataLoader:
    @dataclass
    class Datafile:
        name: str
        path: str

    class InputTimeScale(Enum):
        us = 1
        ms = 2
        sec = 3

    _TIMESCALE_TO_US = {
        InputTimeScale.us: MICROSECOND,
        InputTimeScale.ms: MILLISECOND,
        InputTimeScale.sec: SECOND,
    }

    @staticmethod
    def _window_to_us(window: int, timescale: InputTimeScale) -> int:
        return window * ParquetDataLoader._TIMESCALE_TO_US[timescale]

    def _load_part(self, path: str, ts_from: int, ts_to: int) -> pl.DataFrame:
        return (
            self._lf_by_path[path]
            .filter(
                (pl.col("timestamp") >= ts_from) &
                (pl.col("timestamp") < ts_to)
            )
            .collect()
        )

    @staticmethod
    def _first_ts(lf: pl.LazyFrame) -> int:
        return int(lf.select(pl.col("timestamp").min()).collect()[0, 0])

    @staticmethod
    def _last_ts(lf: pl.LazyFrame) -> int:
        return int(lf.select(pl.col("timestamp").max()).collect()[0, 0])

    def __init__(
        self,
        datafiles: List[Datafile],
        window: int,
        timescale: InputTimeScale,
        until: Optional[int] = None,
    ) -> None:
        self._datafiles = datafiles
        self._time_window_us = self._window_to_us(window, timescale)

        self._lf_by_path = {
            datafile.path: pl.scan_parquet(datafile.path)
            for datafile in datafiles
        }
        self._first_ts_by_name = {
            datafile.name: self._first_ts(self._lf_by_path[datafile.path])
            for datafile in datafiles
        }
        self._last_ts_by_name = {
            datafile.name: self._last_ts(self._lf_by_path[datafile.path])
            for datafile in datafiles
        }

        self._data_start_ts = min(self._first_ts_by_name.values())
        file_end_ts = max(self._last_ts_by_name.values()) + 1
        if until is None:
            self._data_end_ts = file_end_ts
        else:
            until_us = self._window_to_us(
                until,
                timescale
            )
            self._data_end_ts = min(file_end_ts, until_us)

        self._ts_from = self._data_start_ts

    def get_particular_time(self, from_ts: int, to_ts: int) -> Dict[str, pl.DataFrame]:
        return {
            datafile.name: self._load_part(datafile.path, from_ts, to_ts)
            for datafile in self._datafiles
        }

    def get_first_two_weeks(self) -> Dict[str, pl.DataFrame]:
        from_ts = self._data_start_ts
        to_ts = from_ts + TWO_WEEKS_SEC * SECOND
        return self.get_particular_time(from_ts, to_ts)

    def get_first_day(self) -> Dict[str, pl.DataFrame]:
        from_ts = self._data_start_ts
        to_ts = from_ts + 24 * 60 * 60 * SECOND
        return self.get_particular_time(from_ts, to_ts)

    @property
    def data_start_ts(self) -> int:
        return self._data_start_ts

    @property
    def data_end_ts(self) -> int:
        return self._data_end_ts

    def __iter__(self):
        return self

    def __next__(self) -> Dict[str, pl.DataFrame]:
        if self._ts_from >= self._data_end_ts:
            raise StopIteration

        ts_to = min(self._ts_from + self._time_window_us, self._data_end_ts)

        chunk: Dict[str, pl.DataFrame] = {}
        for datafile in self._datafiles:
            chunk[datafile.name] = self._load_part(
                datafile.path,
                ts_from=self._ts_from,
                ts_to=ts_to,
            )

        self._ts_from = ts_to
        return chunk


**`ValidationErrorReason`** - Enum of outcomes from `validate()`: NaNs/Infs, lookahead (`max_used_ts > trades_ts`), or misaligned feature timestamps.


In [135]:
class ValidationErrorReason(Enum):
    HAS_NANs = 1
    HAS_INFs = 2
    HAS_FORWARD_LOOKING = 3
    INCORRECNT_TRADES_TO_TS_ALIGNMENT = 4
    OK = 5

def _validate_nans(data: np.ndarray) -> bool:
    data = np.asarray(data)
    return bool(not np.isnan(data).any())

def _validate_infs(data: np.ndarray) -> bool:
    data = np.asarray(data)
    return bool(not np.isinf(data).any())

def _validate_forward_looking(trades_ts: np.ndarray, max_used_ts: np.ndarray) -> bool:
    trades_ts = np.asarray(trades_ts)
    max_used_ts = np.asarray(max_used_ts)
    return bool(np.all(max_used_ts <= trades_ts))

def _validate_incorrect_aligments(trades_ts, feature_ts):
    trades_ts = np.asarray(trades_ts)
    feature_ts = np.asarray(feature_ts)
    return bool(np.all(trades_ts == feature_ts))

def validate(**data_to_validate) -> ValidationErrorReason:
    if not _validate_nans(data_to_validate["features"]):
        return ValidationErrorReason.HAS_NANs
    if not _validate_infs(data_to_validate["features"]):
        return ValidationErrorReason.HAS_INFs
    if not _validate_forward_looking(data_to_validate["trades_ts"], data_to_validate["max_used_ts"]):
        return ValidationErrorReason.HAS_FORWARD_LOOKING
    if not _validate_incorrect_aligments(data_to_validate["trades_ts"], data_to_validate["features_ts"]):
        return ValidationErrorReason.INCORRECNT_TRADES_TO_TS_ALIGNMENT
    return ValidationErrorReason.OK


In [136]:
binance_btc_bbo = "binance_booktickers_btc"
binance_btc_liq = "binance_liquidations_btc"
binance_btc_trades = "binance_trades_btc"
bybit_btc_liq = "bybit_liquidations_btc"

parquet_names: List[str] = [
    binance_btc_bbo,
    binance_btc_liq,
    binance_btc_trades,
    bybit_btc_liq,
]

parquet_paths: List[str] = [
    "liquidation_task/data/binance_booktickers/perp_btcusdt.parquet",
    "liquidation_task/data/binance_liquidations/perp_btcusdt.parquet",
    "liquidation_task/data/binance_trades/perp_btcusdt.parquet",
    "liquidation_task/data/bybit_liquidations/btcusdt.parquet",
]

datafiles: List[ParquetDataLoader.Datafile] = [
    ParquetDataLoader.Datafile(name, path) 
    for name, path in 
    zip(parquet_names, parquet_paths)
]


**`mark_trades_via_pnl`** - Binary labels: whether mid-price PnL after 30s / 120s / 300s is positive (bps).


In [137]:
def mark_trades_via_pnl(trades, bbo) -> np.ndarray:
    trade_ts = trades["timestamp"].to_numpy()
    trade_px = trades["price"].to_numpy().astype(np.float64)
    side = trades["side"].to_numpy()

    if side.dtype.kind in "OUS":
        if not np.isin(side, ["buy", "sell"]).all():
            raise ValueError("trades['side'] должен содержать только buy/sell")
        sign = np.where(side == "buy", 1.0, -1.0)
    else:
        sign = np.sign(side.astype(np.float64))
        if (sign == 0).any():
            raise ValueError("trades['side'] должен содержать только +/-1")

    bbo_ts = bbo["timestamp"].to_numpy()
    if "mid_price" in bbo.columns:
        bbo_mid = bbo["mid_price"].to_numpy().astype(np.float64)
    else:
        bbo_mid = (
            bbo["bid_price"].to_numpy().astype(np.float64)
            + bbo["ask_price"].to_numpy().astype(np.float64)
        ) / 2.0

    out = np.zeros((len(trade_ts), 3), dtype=np.int8)
    if len(bbo_ts) == 0:
        return out

    for col, tau_sec in enumerate([30, 120, 300]):
        target_ts = trade_ts + tau_sec * SECOND
        idx = np.searchsorted(bbo_ts, target_ts, side="right") - 1
        valid = (idx >= 0) & (target_ts <= bbo_ts[-1])
        pnl = (
            -sign[valid]
            * (bbo_mid[idx[valid]] - trade_px[valid])
            / trade_px[valid]
            * 10_000.0
            + 0.5
        )
        out[valid, col] = (pnl > 0).astype(np.int8)
    return out


# Features


**`Feature`** - Abstract base for trade-aligned signals. Subclasses implement `calculate()` returning `(features, trades_ts, max_used_ts)` per trade.


In [138]:
class Feature(ABC):
    def __init__(self, name: str):
        self._name = name

    @property
    def name(self) -> str:
        return self._name

    @abstractmethod
    def calculate(self, **data) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        raise NotImplemented

    @abstractmethod
    def calculate_max_used_ts(self, **data) -> np.ndarray:
        raise NotImplemented


# Liquidation features

Bybit liquidations are shifted by `+200ms` before feature calculation to account for cross-exchange data latency.


**`LiquidationFeature`** - Sums buy/sell liquidation notionals in `(t_trade - window, t_trade]`. Tracks the latest liquidation timestamp used for leakage checks.


In [139]:
class LiquidationFeature(Feature):
    def __init__(self, name: str):
        super().__init__(name)

    def _prepare_window_notionals(self, **data) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        liquidations = data["liquidations"]
        trades = data["trades"]
        window = data["window_sec"]
        timestamp_shift_us = int(data.get("timestamp_shift_us", 0))

        liq_ts = liquidations["timestamp"].to_numpy().astype(np.int64)
        if timestamp_shift_us != 0:
            liq_ts = liq_ts + timestamp_shift_us

        trades_ts = trades["timestamp"].to_numpy()
        notionals = (liquidations["price"] * liquidations["amount"]).to_numpy().astype(np.float64)

        buy_cumm = np.r_[0.0, np.cumsum(np.where(liquidations["side"] == "buy", notionals, 0.0), dtype=np.float64)]
        sell_cumm = np.r_[0.0, np.cumsum(np.where(liquidations["side"] == "sell", notionals, 0.0), dtype=np.float64)]

        left = np.searchsorted(liq_ts, trades_ts - window, side="left")
        right = np.searchsorted(liq_ts, trades_ts, side="right")

        buy_notional = buy_cumm[right] - buy_cumm[left]
        sell_notional = sell_cumm[right] - sell_cumm[left]
        return buy_notional, sell_notional, trades_ts, liq_ts, right, left

    def calculate_max_used_ts(self, **data) -> np.ndarray:
        trades_ts = data["trades_ts"]
        liq_ts = data["liq_ts"]
        most_recent = data["most_recent"]
        least_recent = data["least_recent"]

        max_used_ts = trades_ts.copy()
        mask = most_recent > least_recent
        max_used_ts[mask] = liq_ts[most_recent[mask] - 1]
        return max_used_ts


**`LiqudationClusterImbalance`** - Normalized liquidation imbalance `(buy - sell) / (buy + sell)`, zero when the window is empty.


In [140]:
class LiqudationClusterImbalance(LiquidationFeature):
    def __init__(self, name: str = "liquidation_notional_imbalance"):
        super().__init__(name)

    def calculate(self, **data) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        buy_notional, sell_notional, trades_ts, liq_ts, right, left = self._prepare_window_notionals(**data)

        denom = buy_notional + sell_notional
        imbalance = np.divide(
            buy_notional - sell_notional,
            denom,
            out=np.zeros_like(denom, dtype=np.float64),
            where=denom > 0,
        )
        feature = imbalance.astype(np.float32).reshape((-1, 1))

        max_used_ts = self.calculate_max_used_ts(
            trades_ts=trades_ts,
            liq_ts=liq_ts,
            most_recent=right,
            least_recent=left,
        )
        return feature, trades_ts, max_used_ts


**`LiqudationClusterStrength`** - Signed log-strength of liquidation flow: `sign(buy - sell) * log1p(|buy - sell|)`.


In [141]:
class LiqudationClusterStrength(LiquidationFeature):
    def __init__(self, name: str = "liquidation_notional_strength"):
        super().__init__(name)

    def calculate(self, **data) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        buy_notional, sell_notional, trades_ts, liq_ts, right, left = self._prepare_window_notionals(**data)

        delta = buy_notional - sell_notional
        signed_log_delta = np.sign(delta) * np.log1p(np.abs(delta))
        feature = signed_log_delta.astype(np.float32).reshape((-1, 1))

        max_used_ts = self.calculate_max_used_ts(
            trades_ts=trades_ts,
            liq_ts=liq_ts,
            most_recent=right,
            least_recent=left,
        )
        return feature, trades_ts, max_used_ts


# BBO features


**`BboFeature`** - Shared BBO windowing and snapshot indices aligned to each trade. Computes `max_used_ts` from the newest BBO tick referenced.


In [142]:
class BboFeature(Feature):
    def __init__(self, name: str):
        super().__init__(name)

    def _prepare_bbo_window(self, **data) -> Dict[str, np.ndarray]:
        bbo = data["bbo"]
        trades = data["trades"]
        window = data["window_sec"]

        bbo_ts = bbo["timestamp"].to_numpy()
        trades_ts = trades["timestamp"].to_numpy()

        bid_price = bbo["bid_price"].to_numpy().astype(np.float64)
        ask_price = bbo["ask_price"].to_numpy().astype(np.float64)
        bid_amount = bbo["bid_amount"].to_numpy().astype(np.float64)
        ask_amount = bbo["ask_amount"].to_numpy().astype(np.float64)
        mid_price = (bid_price + ask_price) / 2.0

        left = np.searchsorted(bbo_ts, trades_ts - window, side="left")
        right = np.searchsorted(bbo_ts, trades_ts, side="right")
        snap_idx = right - 1
        snap_valid = snap_idx >= 0

        return {
            "bbo_ts": bbo_ts,
            "trades_ts": trades_ts,
            "bid_price": bid_price,
            "ask_price": ask_price,
            "bid_amount": bid_amount,
            "ask_amount": ask_amount,
            "mid_price": mid_price,
            "left": left,
            "right": right,
            "snap_idx": snap_idx,
            "snap_valid": snap_valid,
        }

    def calculate_max_used_ts(self, **data) -> np.ndarray:
        trades_ts = data["trades_ts"]
        bbo_ts = data["bbo_ts"]
        most_recent = data["most_recent"]
        least_recent = data.get("least_recent", most_recent)
        mask = data.get("mask", most_recent > least_recent)

        max_used_ts = trades_ts.copy()
        max_used_ts[mask] = bbo_ts[most_recent[mask] - 1]
        return max_used_ts


**`BboSpreadBps`** - Bid-ask spread at the last BBO before the trade, in basis points of mid.


In [143]:
class BboSpreadBps(BboFeature):
    def __init__(self):
        super().__init__("bbo_spread_bps")

    def calculate(self, **data) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        ctx = self._prepare_bbo_window(**data)

        feature = np.zeros_like(ctx["trades_ts"], dtype=np.float64)
        valid = ctx["snap_valid"]
        if np.any(valid):
            idx = ctx["snap_idx"][valid]
            spread = ctx["ask_price"][idx] - ctx["bid_price"][idx]
            mid = ctx["mid_price"][idx]
            spread_bps = np.divide(
                spread,
                mid,
                out=np.zeros_like(spread, dtype=np.float64),
                where=mid != 0,
            ) * 10_000.0
            feature[valid] = spread_bps

        feature = feature.astype(np.float32).reshape((-1, 1))
        max_used_ts = self.calculate_max_used_ts(
            trades_ts=ctx["trades_ts"],
            bbo_ts=ctx["bbo_ts"],
            most_recent=ctx["snap_idx"] + 1,
            least_recent=ctx["snap_idx"],
            mask=valid,
        )
        return feature, ctx["trades_ts"], max_used_ts


**`BboVolumeImbalance`** - Top-of-book size imbalance `(bid_amount - ask_amount) / (bid_amount + ask_amount)` on the snapshot.


In [144]:
class BboVolumeImbalance(BboFeature):
    def __init__(self):
        super().__init__("bbo_volume_imbalance")

    def calculate(self, **data) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        ctx = self._prepare_bbo_window(**data)

        feature = np.zeros_like(ctx["trades_ts"], dtype=np.float64)
        valid = ctx["snap_valid"]
        if np.any(valid):
            idx = ctx["snap_idx"][valid]
            bid_amt = ctx["bid_amount"][idx]
            ask_amt = ctx["ask_amount"][idx]
            denom = bid_amt + ask_amt
            imbalance = np.divide(
                bid_amt - ask_amt,
                denom,
                out=np.zeros_like(denom, dtype=np.float64),
                where=denom > 0,
            )
            feature[valid] = imbalance

        feature = feature.astype(np.float32).reshape((-1, 1))
        max_used_ts = self.calculate_max_used_ts(
            trades_ts=ctx["trades_ts"],
            bbo_ts=ctx["bbo_ts"],
            most_recent=ctx["snap_idx"] + 1,
            least_recent=ctx["snap_idx"],
            mask=valid,
        )
        return feature, ctx["trades_ts"], max_used_ts


**`BboMidSmoothDeltaBps`** - Current mid vs. mean mid over the lookback window, in basis points.


In [145]:
class BboMidSmoothDeltaBps(BboFeature):
    def __init__(self):
        super().__init__("bbo_mid_smooth_delta_bps")

    def calculate(self, **data) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        ctx = self._prepare_bbo_window(**data)

        mid_cumm = np.r_[0.0, np.cumsum(ctx["mid_price"], dtype=np.float64)]
        counts = ctx["right"] - ctx["left"]
        mid_sum = mid_cumm[ctx["right"]] - mid_cumm[ctx["left"]]
        mid_smooth = np.divide(
            mid_sum,
            counts,
            out=np.zeros_like(mid_sum, dtype=np.float64),
            where=counts > 0,
        )

        feature = np.zeros_like(ctx["trades_ts"], dtype=np.float64)
        valid = ctx["snap_valid"] & (counts > 0)
        if np.any(valid):
            idx = ctx["snap_idx"][valid]
            mid_now = ctx["mid_price"][idx]
            mid_ref = mid_smooth[valid]
            delta_bps = np.divide(
                mid_now - mid_ref,
                mid_ref,
                out=np.zeros_like(mid_now, dtype=np.float64),
                where=mid_ref != 0,
            ) * 10_000.0
            feature[valid] = delta_bps

        feature = feature.astype(np.float32).reshape((-1, 1))
        max_used_ts = self.calculate_max_used_ts(
            trades_ts=ctx["trades_ts"],
            bbo_ts=ctx["bbo_ts"],
            most_recent=ctx["right"],
            least_recent=ctx["left"],
        )
        return feature, ctx["trades_ts"], max_used_ts


**`FeatureAnalyzer`** - Abstract metric that maps feature values (and optionally targets) to one float.


In [146]:
class FeatureAnalyzer(ABC):
    def __init__(self, name: str) -> None:
        self._name = name

    @abstractmethod
    def calculate(self, *data) -> float:
        pass

    @property
    def name(self) -> str:
        return self._name


**`EntropyAnalyzer`** - Shannon entropy (bits) of the empirical distribution of feature values.


In [147]:
class EntropyAnalyzer(FeatureAnalyzer):
    def __init__(self):
        super().__init__("entropy")

    def calculate(self, feature: np.ndarray) -> float:

        _, counts = np.unique(feature, return_counts=True)
        probs = counts / counts.sum()
        return float(scipy_entropy(probs, base=2))


**`SpearmanCorrelationAnalyzer`** - Spearman rank correlation between the feature and the target.


In [148]:
class SpearmanCorrelationAnalyzer(FeatureAnalyzer):
    def __init__(self):
        super().__init__("spearman_correlation")

    def calculate(
        self,
        feature: np.ndarray,
        target: np.ndarray,
    ) -> float:
        corr, _ = spearmanr(feature, target)
        return float(corr)


**`MutualInformationAnalyzer`** - Nonparametric mutual information estimate between feature and target.


In [149]:
class MutualInformationAnalyzer(FeatureAnalyzer):
    def __init__(self):
        super().__init__("mutual_information")

    def calculate(
        self,
        feature: np.ndarray,
        target: np.ndarray,
    ) -> float:
        mi = mutual_info_regression(
            feature.reshape(-1, 1),
            target,
            random_state=42,
        )

        return float(mi[0])


**`KSAnalyzer`** - Kolmogorov-Smirnov distance between two feature samples (e.g. target 0 vs. 1).


In [150]:
class KSAnalyzer(FeatureAnalyzer):
    def __init__(self):
        super().__init__("ks")

    def calculate(
        self,
        expected: np.ndarray,
        actual: np.ndarray,
    ) -> float:

        statistic, _ = ks_2samp(
            expected,
            actual,
        )

        return float(statistic)


**`ModelBasedFeatureAnalyzer`** - Fits a model X->y and returns an in-sample score; reshapes 1D X to `(N, 1)` automatically.


In [151]:
class ModelBasedFeatureAnalyzer(FeatureAnalyzer):
    def __init__(self, name: str):
        super().__init__(name)
        self._model = None

    @abstractmethod
    def _fit(self, X: np.ndarray, y: np.ndarray):
        pass

    @abstractmethod
    def _score(self, X: np.ndarray, y: np.ndarray) -> float:
        pass

    def calculate(self, X: np.ndarray, y: np.ndarray) -> float:
        X = np.asarray(X)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        self._fit(X, y)
        return float(self._score(X, y))


**`LinearRegressionAnalyzer`** - Ordinary least squares on the feature; returns training-set R2.


In [152]:
class LinearRegressionAnalyzer(ModelBasedFeatureAnalyzer):
    def __init__(self):
        super().__init__("linear_regression_r2")

    def _fit(self, X: np.ndarray, y: np.ndarray):
        self._model = LinearRegression()
        self._model.fit(X, y)

    def _score(self, X: np.ndarray, y: np.ndarray) -> float:
        preds = self._model.predict(X)
        return r2_score(y, preds)


**`TreeR2Analyzer`** - Shallow random forest on the feature; returns training-set R2.


In [153]:
class TreeR2Analyzer(ModelBasedFeatureAnalyzer):
    def __init__(self):
        super().__init__("tree_r2")

    def _fit(self, X: np.ndarray, y: np.ndarray):
        self._model = RandomForestRegressor(
            n_estimators=50,
            max_depth=5,
            random_state=42,
        )
        self._model.fit(X, y)

    def _score(self, X: np.ndarray, y: np.ndarray) -> float:
        preds = self._model.predict(X)
        return r2_score(y, preds)


# Runs


## Data loading


In [154]:
pdl = ParquetDataLoader(datafiles, TWO_WEEKS_SEC, ParquetDataLoader.InputTimeScale.sec)
train = pdl.get_first_day()
train[binance_btc_trades].head()


timestamp,ticker,side,price,amount
i64,str,str,f64,f64
1764547200047000,"""perp:btcusdt""","""sell""",90320.5,0.003
1764547200047000,"""perp:btcusdt""","""sell""",90320.5,0.003
1764547200047000,"""perp:btcusdt""","""sell""",90320.5,0.003
1764547200047000,"""perp:btcusdt""","""sell""",90320.5,0.419
1764547200050000,"""perp:btcusdt""","""buy""",90320.6,0.034


## Targets


In [155]:
labels = mark_trades_via_pnl(train[binance_btc_trades], train[binance_btc_bbo])


## Feature pipeline


In [ ]:
window_sec = 30 * SECOND

feature_pipeline = [
    (
        LiqudationClusterImbalance(),
        {
            "liquidations": train[binance_btc_liq],
            "trades": train[binance_btc_trades],
            "window_sec": window_sec,
        },
    ),
    (
        LiqudationClusterStrength(),
        {
            "liquidations": train[binance_btc_liq],
            "trades": train[binance_btc_trades],
            "window_sec": window_sec,
        },
    ),
    (
        LiqudationClusterImbalance("bybit_liquidation_notional_imbalance"),
        {
            "liquidations": train[bybit_btc_liq],
            "trades": train[binance_btc_trades],
            "window_sec": window_sec,
            "timestamp_shift_us": BYBIT_LIQUIDATION_DELAY_US,
        },
    ),
    (
        LiqudationClusterStrength("bybit_liquidation_notional_strength"),
        {
            "liquidations": train[bybit_btc_liq],
            "trades": train[binance_btc_trades],
            "window_sec": window_sec,
            "timestamp_shift_us": BYBIT_LIQUIDATION_DELAY_US,
        },
    ),
    (
        BboSpreadBps(),
        {
            "bbo": train[binance_btc_bbo],
            "trades": train[binance_btc_trades],
            "window_sec": window_sec,
        },
    ),
    (
        BboVolumeImbalance(),
        {
            "bbo": train[binance_btc_bbo],
            "trades": train[binance_btc_trades],
            "window_sec": window_sec,
        },
    ),
    (
        BboMidSmoothDeltaBps(),
        {
            "bbo": train[binance_btc_bbo],
            "trades": train[binance_btc_trades],
            "window_sec": window_sec,
        },
    ),
]


## Feature computation


In [ ]:
target = labels[:, 0].astype(np.float64) 

res = {}

for f in feature_pipeline:
    res[f[0].name] = f[0].calculate(**f[1])


## Feature analysis

In [ ]:
tra = TreeR2Analyzer()
scores = {}

for name, (feat, _, _) in res.items():
    scores[name] = tra.calculate(feat[:, 0], target)

display(scores)

In [ ]:
lra = LinearRegressionAnalyzer()
scores = {}

for name, (feat, _, _) in res.items():
    scores[name] = lra.calculate(feat[:, 0], target)

display(scores)


In [160]:
entropy_analyzer = EntropyAnalyzer()

for name, (feat, _, _) in res.items():
    display(name, entropy_analyzer.calculate(feat[:, 0]))


'liquidation_notional_imbalance'

2.1638028844798884

'liquidation_notional_strength'

6.8407871912518186

'bybit_liquidation_notional_imbalance'

...

14.001525844225837

'bbo_volume_imbalance'

15.947809013029708

'bbo_mid_smooth_delta_bps'

16.254305565303238

In [161]:
mutual_info_analyzer = MutualInformationAnalyzer()

for name, (feat, _, _) in res.items():
    display(name, mutual_info_analyzer.calculate(feat[:, 0], target))


'liquidation_notional_imbalance'

0.00794303448649103

'liquidation_notional_strength'

0.05487509910888555

'bybit_liquidation_notional_imbalance'

...

0.24960631234532293

'bbo_volume_imbalance'

0.47311955771613334

'bbo_mid_smooth_delta_bps'

0.4860274425909168

In [162]:
ks_analyzer = KSAnalyzer()

for name, (feat, _, _) in res.items():
    display(name, ks_analyzer.calculate(feat[:, 0], target))


'liquidation_notional_imbalance'

0.4531716243158979

'liquidation_notional_strength'

0.4531716243158979

'bybit_liquidation_notional_imbalance'

...

0.5031441096971887

'bbo_volume_imbalance'

0.49674842439083133

'bbo_mid_smooth_delta_bps'

0.5696370746524293